### 1) Imports

In [1]:
import pandas as pd
import numpy as np

### 2) Load data and inspect

In [2]:
app = pd.read_csv('../data/application_train.csv')
bureau = pd.read_csv('../data/bureau.csv')

print("application_train shape:", app.shape)
print("bureau shape:", bureau.shape)

display(app.head())
display(app.info())

application_train shape: (307511, 122)
bureau shape: (1716428, 17)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


<class 'pandas.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 122 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float64(65), int64(41), str(16)
memory usage: 286.2 MB


None

### 3) Basic cleanup

In [3]:
# Make sure column names are clean
app.columns = app.columns.str.strip()
bureau.columns = bureau.columns.str.strip()

# Create a copy so we do not overwrite the raw file
df = app.copy()

### 4) Fix sentinel values

In [4]:
# DAYS_EMPLOYED uses 365243 as a fake value for unemployment/retirement
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

# Optional: create a readable age column
df['age'] = (-df['DAYS_BIRTH']) / 365

### 5) Missing data report

In [5]:
missing_count = df.isna().sum()
missing_pct = (missing_count / len(df) * 100).round(2)

missing_report = (
    pd.DataFrame({
        'missing_count': missing_count,
        'missing_pct': missing_pct
    })
    .sort_values('missing_pct', ascending=False)
)

display(missing_report.head(25))

print("Features with >50% missing:", (missing_report['missing_pct'] > 50).sum())

,missing_count,missing_pct
COMMONAREA_AVG,214865,69.87
COMMONAREA_MEDI,214865,69.87
COMMONAREA_MODE,214865,69.87
NONLIVINGAPARTMENTS_MODE,213514,69.43
NONLIVINGAPARTMENTS_MEDI,213514,69.43
NONLIVINGAPARTMENTS_AVG,213514,69.43
FONDKAPREMONT_MODE,210295,68.39
LIVINGAPARTMENTS_MEDI,210199,68.35
LIVINGAPARTMENTS_MODE,210199,68.35
LIVINGAPARTMENTS_AVG,210199,68.35


Features with >50% missing: 41


### Some features have a large amount of missing data. In a regulated credit setting, that can happen because certain fields are only collected for specific applicants or product types.

### 6) Create Missingness Indicators

In [6]:
# Columns with more than 5% missing values
cols_with_missing = missing_report[missing_report['missing_pct'] > 5].index.tolist()

# Do not create indicators for ID or target
cols_with_missing = [c for c in cols_with_missing if c not in ['SK_ID_CURR', 'TARGET']]

for col in cols_with_missing:
    df[f'{col}_MISSING'] = df[col].isna().astype(int)

print("Missingness indicator columns created:", len(cols_with_missing))

Missingness indicator columns created: 58


### Missingness itself can be predictive in credit risk. For example, applicants may leave sensitive fields blank, and that pattern can carry risk information.

### 7) Simple imputation

In [7]:
# Separate numeric and categorical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

# Remove TARGET and ID from imputation lists
num_cols = [c for c in num_cols if c not in ['TARGET', 'SK_ID_CURR']]
cat_cols = [c for c in cat_cols if c not in ['TARGET', 'SK_ID_CURR']]

# Fill numeric columns with median
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical columns with mode
for col in cat_cols:
    if df[col].isna().any():
        mode_value = df[col].mode(dropna=True)[0]
        df[col] = df[col].fillna(mode_value)

# Final check
print("Remaining missing values:", df.isna().sum().sum())

C:\Users\humza\AppData\Local\Temp\ipykernel_18548\2800987593.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns.tolist()


Remaining missing values: 0


### 8) Quick review of what was changed

In [8]:
summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_after': df.isna().sum()
})

display(summary.head(20))

,dtype,missing_after
SK_ID_CURR,int64,0
TARGET,int64,0
NAME_CONTRACT_TYPE,str,0
CODE_GENDER,str,0
FLAG_OWN_CAR,str,0
FLAG_OWN_REALTY,str,0
CNT_CHILDREN,int64,0
AMT_INCOME_TOTAL,float64,0
AMT_CREDIT,float64,0
AMT_ANNUITY,float64,0


### 9) Save the cleaned interim file

In [9]:
df.to_csv('../data/interim_train.csv', index=False)
print("Saved: ../data/interim_train.csv")

Saved: ../data/interim_train.csv
